# Pooling & Network Architecture

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/cnns/02-pooling-and-architectures

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Max Pooling

In [ ]:
def max_pool(X, size=2, stride=2):
    H, W = X.shape
    OH = (H - size) // stride + 1
    OW = (W - size) // stride + 1
    out = np.zeros((OH, OW))
    for i in range(OH):
        for j in range(OW):
            out[i, j] = np.max(X[i*stride:i*stride+size, j*stride:j*stride+size])
    return out

feature_map = np.random.randn(8, 8)
pooled = max_pool(feature_map, 2, 2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(feature_map, cmap='magma')
axes[0].set_title(f'Feature Map ({feature_map.shape})', color='white')
axes[1].imshow(pooled, cmap='magma')
axes[1].set_title(f'After MaxPool ({pooled.shape})', color='white')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

## Receptive Field Growth

Stacking 3×3 convolutions increases the receptive field:

In [ ]:
layers = [1, 2, 3, 4, 5]
receptive = [3 + 2 * (l - 1) for l in layers]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(layers, receptive, color='#818cf8', alpha=0.8)
ax.set_xlabel('Number of 3×3 Conv Layers')
ax.set_ylabel('Receptive Field')
ax.set_title('Receptive Field Grows with Depth', color='white')
for l, r in zip(layers, receptive):
    ax.text(l, r + 0.3, f'{r}×{r}', ha='center', color='#94a3b8', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Why VGG uses only 3x3: two 3x3 == one 5x5 receptive field, with fewer params.
# Receptive field of L stacked f x f stride-1 convs:  RF = 1 + L*(f-1)
def rf(L, f): return 1 + L * (f - 1)
print('RF of two 3x3 =', rf(2, 3), ' vs one 5x5 =', rf(1, 5), ' (equal)')
print('RF of three 3x3 =', rf(3, 3), ' vs one 7x7 =', rf(1, 7), ' (equal)\n')

# Parameter count for C in- and out-channels: f*f*C*C per conv
def params(f, C, n=1): return n * f * f * C * C
for C in [32, 64]:
    one5 = params(5, C, 1)
    two3 = params(3, C, 2)
    print(f'C={C}: one 5x5 = {one5:,}   two 3x3 = {two3:,}   '
          f'saving = {one5 - two3:,} ({100*(one5-two3)/one5:.0f}% fewer)')
print('\n-> same receptive field, ~28% fewer parameters, plus an extra ReLU non-linearity.')


## Max vs average pooling

Pooling downsamples each window. **Max** keeps the strongest activation (edges/texture); **average** smooths. **Global average pooling** collapses each channel to one number.

In [ ]:
fmap = np.array([[1, 3, 2, 4],
                 [5, 6, 1, 2],
                 [7, 2, 3, 0],
                 [1, 2, 4, 8]], dtype=float)

def pool(x, k=2, mode='max'):
    H, W = x.shape
    out = np.zeros((H // k, W // k))
    for i in range(0, H, k):
        for j in range(0, W, k):
            win = x[i:i+k, j:j+k]
            out[i//k, j//k] = win.max() if mode == 'max' else win.mean()
    return out

print('max pool:\n', pool(fmap, 2, 'max'))
print('avg pool:\n', pool(fmap, 2, 'avg'))
print('global avg pool:', fmap.mean())

## Key takeaways

- **Pooling** shrinks spatial size, adds translation invariance, and cuts compute.
- Architectures evolved: LeNet → AlexNet → VGG (deep 3×3) → ResNet (**skip connections**).
- **Skip connections** let gradients bypass layers, enabling 100+ layer networks.
- Global average pooling replaces giant fully-connected heads in modern CNNs.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — 2×2 max pooling

Max pooling keeps the strongest activation in each block — shrinking the map 2× per side and buying a little translation tolerance. Implement the standard 2×2, stride-2 version. The last check demonstrates the tolerance: moving the strongest value *within its block* leaves the pooled output unchanged.

In [ ]:
def max_pool(img):
    """2x2 max pooling with stride 2 (assume even dimensions)."""
    img = np.asarray(img, dtype=float)
    H, W = img.shape
    out = np.zeros((H // 2, W // 2))

    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            # TODO(you): max of the 2x2 block at rows 2i:2i+2, cols 2j:2j+2
            out[i, j] = ...

    return out

In [ ]:
# Checks — run me
img = np.array([
    [1, 3, 2, 0],
    [4, 2, 1, 1],
    [0, 1, 8, 2],
    [2, 1, 0, 3],
])
assert max_pool(img).shape == (2, 2), "4x4 -> 2x2"
assert np.allclose(max_pool(img), [[4, 2], [2, 8]]), "max of each 2x2 block"

shifted = img.copy(); shifted[0, 0], shifted[0, 1] = 3, 1   # move values within the top-left block
assert np.allclose(max_pool(shifted), max_pool(img)), \
    "values moved within their block -> pooled output unchanged (translation tolerance)"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def max_pool(img):
    img = np.asarray(img, dtype=float)
    H, W = img.shape
    out = np.zeros((H // 2, W // 2))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = np.max(img[2 * i:2 * i + 2, 2 * j:2 * j + 2])
    return out
```

</details>

### Exercise 2 — Receptive-field growth

How much of the input does one output pixel *see*? Walk the layers forward, tracking the receptive field and the cumulative stride ("jump"):

$$rf \mathrel{+}= (k - 1) \cdot jump, \qquad jump \mathrel{*}= s$$

starting from $rf = jump = 1$. The checks verify VGG's famous economy: **three stacked 3×3 convs see exactly as far as one 7×7** — with fewer parameters and two extra nonlinearities — and that a stride-2 pool doubles the growth rate of everything after it.

In [ ]:
def receptive_field(layers):
    """Receptive field of the final output, given layers as (kernel, stride) pairs."""
    rf, jump = 1, 1
    for k, s in layers:
        # TODO(you): grow the receptive field by (k - 1) * jump
        rf = ...

        # TODO(you): multiply the jump by this layer's stride
        jump = ...

    return rf

In [ ]:
# Checks — run me
assert receptive_field([(3, 1)]) == 3, "one 3x3 conv sees 3x3"
assert receptive_field([(3, 1), (3, 1)]) == 5, "two stacked 3x3 convs see 5x5"
assert receptive_field([(3, 1), (3, 1), (3, 1)]) == 7, "three see 7x7 — same as one 7x7, fewer params"
assert receptive_field([(7, 1)]) == receptive_field([(3, 1), (3, 1), (3, 1)]), "VGG's trick"
assert receptive_field([(3, 1), (2, 2), (3, 1)]) == 8, "a stride-2 pool doubles later layers' growth"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def receptive_field(layers):
    rf, jump = 1, 1
    for k, s in layers:
        rf = rf + (k - 1) * jump
        jump = jump * s
    return rf
```

</details>